# Session 2: building a part from distance functions
**A. Real-world problem.** A designer wants a mug that holds 300 mL. The wall thickness and the
radius are free; the height is fixed by the shelf. How do we build the mug from simple shapes,
let a program change its size, and compute the volume? Lengths are in cm, so 1 cm^3 = 1 mL.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
HANDLE_R, HANDLE_T = 2.0, 0.45     # handle ring: centre-line radius and tube radius (cm)

## Step 1: shapes and the CAD operations
With signed distances `a` and `b` of two shapes:

| operation | formula |
|---|---|
| union | `min(a, b)` |
| intersection | `max(a, b)` |
| subtraction (a minus b) | `max(a, -b)` |
| offset (grow by t) | `a - t` |
| shell (a wall of thickness t on each side of the surface) | `abs(a) - t` |

In [ ]:
def sd_circle(p, c, r):
    return np.linalg.norm(np.asarray(p, float) - np.asarray(c, float), axis=-1) - r


def sd_box(p, c, half):
    q = np.abs(np.asarray(p, float) - np.asarray(c, float)) - np.asarray(half, float)
    return np.linalg.norm(np.maximum(q, 0.0), axis=-1) + np.minimum(np.max(q, axis=-1), 0.0)


def op_union(a, b):
    return np.minimum(a, b)


def op_intersect(a, b):
    return np.maximum(a, b)


def op_subtract(a, b):
    """a minus b: keep the points that are in a and not in b."""
    return np.maximum(a, -b)


def op_shell(a, t):
    return np.abs(a) - t

## Step 2: smooth union
`min` leaves a sharp crease where two shapes meet. A *smooth minimum* rounds it with a blend
radius `k` (cm). We use the quadratic polynomial version:

    h = max(k - |a - b|, 0) / k
    smin(a, b, k) = min(a, b) - h * h * k / 4

When the shapes are far apart (`|a - b| >= k`), `h = 0` and `smin` is exactly `min`. When the
two distances are equal, the surface is pulled outward by the largest amount, `k / 4`.

In [ ]:
def smin(a, b, k):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if k <= 0:
        return np.minimum(a, b)
    h = np.maximum(k - np.abs(a - b), 0.0) / k
    return np.minimum(a, b) - h * h * k / 4.0

## Step 3: a parametric mug cross-section
We cut the mug in half along its axis and look at the plane `(x, z)`: `x` is across the mug and `z`
is up. The outer wall is a box of half-width `R` from `z = 0` to `z = H`. The cavity is a box of
half-width `R - t` that starts at the floor `z = t` and reaches above the rim (open top). The
handle is a ring (a torus cut open by the plane) found as `|circle| - tube radius`, kept only
outside the wall, and blended to the body with radius `k`.

In [ ]:
def body2d(p, R, H, t):
    """Mug body without the handle: outer box minus the cavity box."""
    p = np.asarray(p, float)
    outer = sd_box(p, (0.0, H / 2.0), (R, H / 2.0))
    cavity = sd_box(p, (0.0, (t + H + 1.0) / 2.0), (R - t, (H + 1.0 - t) / 2.0))
    return op_subtract(outer, cavity)


def handle2d(p, R, H, t):
    """Handle ring attached to the +x side of the wall, clipped so it never enters the cavity."""
    p = np.asarray(p, float)
    ring = np.abs(sd_circle(p, (R + 1.0, 0.55 * H), HANDLE_R)) - HANDLE_T
    return op_intersect(ring, (R - t / 2.0) - p[..., 0])


def mug2d(p, R, H, t, k=0.0):
    return smin(body2d(p, R, H, t), handle2d(p, R, H, t), k)


def area_by_cells(f, lo, hi, n):
    """Area of {f < 0} inside the box lo..hi, by counting n by n cell centres."""
    x = lo[0] + (np.arange(n) + 0.5) * (hi[0] - lo[0]) / n
    z = lo[1] + (np.arange(n) + 0.5) * (hi[1] - lo[1]) / n
    X, Z = np.meshgrid(x, z)
    inside = f(np.stack([X, Z], axis=-1)) < 0
    return float(inside.mean() * (hi[0] - lo[0]) * (hi[1] - lo[1]))


def body_area(R, H, t):
    """Analytic cross-section area of the body: outer box minus the cavity box."""
    return 2 * R * H - 2 * (R - t) * (H - t)

## Step 4: volume of revolution and the 300 mL design problem
The body is a solid of revolution about the z axis, so it has simple volumes:

    cavity (capacity)  V_c = pi (R - t)^2 (H - t)
    material           V_m = pi R^2 H - V_c

For a wanted capacity V, height H and wall t, solve for the outer radius:
`R = t + sqrt(V / (pi (H - t)))`. Another route: sum `2 pi x dx dz` over the inside cells of the
half cross-section `x >= 0` (a slice rotated about the axis), which we use as a check.

In [ ]:
def capacity(R, H, t):
    return float(np.pi * (R - t) ** 2 * (H - t))


def material_volume(R, H, t):
    return float(np.pi * R ** 2 * H - capacity(R, H, t))


def radius_for_capacity(V, H, t):
    if t <= 0 or t >= H:
        raise ValueError("wall thickness must satisfy 0 < t < H")
    return float(t + np.sqrt(V / (np.pi * (H - t))))


def revolve_volume(f, R, H, n):
    """Volume of the solid made by rotating the half cross-section {f < 0, x >= 0} about the z axis."""
    dx, dz = (R + 0.5) / n, (H + 1.0) / n
    x = (np.arange(n) + 0.5) * dx
    z = -0.5 + (np.arange(n) + 0.5) * dz
    X, Z = np.meshgrid(x, z)
    inside = f(np.stack([X, Z], axis=-1)) < 0
    return float(np.sum(inside * 2 * np.pi * X * dx * dz))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    R, H, t = radius_for_capacity(300.0, 9.0, 0.5), 9.0, 0.5
    print(f"design: H = 9 cm, t = 0.5 cm, capacity 300 mL needs R = {R:.3f} cm")
    print(f"check: capacity({R:.3f}) = {capacity(R, H, t):.2f} mL, material = {material_volume(R, H, t):.1f} cm^3")

    lo, hi = (-R - 1.0, -0.8), (R + 5.0, H + 1.0)
    n = 500
    xs = np.linspace(lo[0], hi[0], n)
    zs = np.linspace(lo[1], hi[1], n)
    X, Z = np.meshgrid(xs, zs)
    P = np.stack([X, Z], axis=-1)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), sharey=True)
    for ax, k, name in zip(axes, [0.0, 1.0], ["sharp union (k = 0)", "smooth union (k = 1 cm)"]):
        d = mug2d(P, R, H, t, k)
        ax.imshow(d, extent=[lo[0], hi[0], lo[1], hi[1]], origin="lower", cmap="RdBu", vmin=-3, vmax=3)
        ax.contour(X, Z, d, levels=[0.0], colors="k", linewidths=2)
        ax.set_aspect("equal")
        ax.set_title(name)
        ax.set_xlabel("x across the mug (cm)")
    axes[0].set_ylabel("z up (cm)")
    fig.tight_layout()
    fig.savefig(OUT / "mug_section.png", dpi=110)

    a = np.linspace(-3, 3, 400)
    fig2, ax2 = plt.subplots(figsize=(6.2, 3.8))
    ax2.plot(a, np.minimum(a, 0.0), "k--", label="min(a, 0)")
    for k in (0.5, 1.0, 2.0):
        ax2.plot(a, smin(a, 0.0, k), label=f"smin, k = {k} cm")
    ax2.set_xlabel("distance a to shape 1 (cm), shape 2 at distance 0")
    ax2.set_ylabel("blended distance (cm)")
    ax2.set_title("Smooth minimum equals min when |a - b| >= k")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "smooth_min.png", dpi=120)

    for nn in (50, 100, 200, 400):
        A = area_by_cells(lambda q: body2d(q, R, H, t), (-R - 0.5, -0.5), (R + 0.5, H + 0.5), nn)
        print(f"n = {nn:4d}: body cross-section area by counting {A:7.3f}  analytic {body_area(R, H, t):7.3f}")
    V = revolve_volume(lambda q: body2d(q, R, H, t), R, H, 400)
    print(f"material by revolving the counted slice: {V:.1f}  analytic {material_volume(R, H, t):.1f}")
    print("handle ring area by counting:", round(area_by_cells(lambda q: np.abs(sd_circle(q, (0, 0), 2.0)) - 0.45,
                                                                 (-3, -3), (3, 3), 600), 3),
          " analytic 4 pi Rh r =", round(4 * np.pi * 2.0 * 0.45, 3))

## Coding tasks
1. **Easy.** Print the capacity of the mug for R = 4.5 cm, t = 0.5 cm, H = 9 cm and compare with the app, then find the radius that gives 250 mL.
2. **Stretch.** Add a lip: make a thicker rim by subtracting a smaller cavity near the top (a second box) in `body2d` and plot it.
3. **Explain.** Explain, with the formulas, why the smooth union with k = 2 cm pulls the surface out by 0.5 cm at the crotch of the handle.

## Assessment
- **A. Conceptual.** Why does the smooth union add material only near the place where the two shapes come close, and not everywhere?
- **B. Numerical.** Find the outer radius for a mug of height 10 cm and wall 0.4 cm that holds 300 mL.
- **C. Graph and simulation interpretation.** In the smooth-min graph with blend radius k = 2 cm, how far below min(a, b) is the blended value at a = b?
- **D. Design and optimisation challenge.** Design: for H = 9 cm and 300 mL, compare walls of 0.4 cm and 0.8 cm. Give the radius and the volume of material for each, and say which you would choose for a mug that is dropped often, and why.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    c0 = (0.0, 0.0)
    # CAD operations on a big circle A (r=2) and a small circle B (r=1) at the same centre-line
    A, B = (0.0, 0.0), (1.5, 0.0)
    pts = np.array([[-1.5, 0.0], [1.4, 0.0], [2.8, 0.0], [5.0, 0.0]])   # in A only, in both, in B only, outside
    a = L.sd_circle(pts, A, 2.0)
    b = L.sd_circle(pts, B, 1.5)
    checks.append(("union is inside where either is inside", list(L.op_union(a, b) < 0) == [True, True, True, False]))
    checks.append(("intersection is inside only where both are inside", list(L.op_intersect(a, b) < 0) == [False, True, False, False]))
    checks.append(("subtraction A minus B keeps the points in A and not in B", list(L.op_subtract(a, b) < 0) == [True, False, False, False]))
    checks.append(("shell: a point on the circle has value -t", abs(float(L.op_shell(L.sd_circle((2, 0), c0, 2.0), 0.3)) + 0.3) < 1e-12))
    checks.append(("shell: a point 0.3 from the surface (either side) is on the shell surface",
                   abs(float(L.op_shell(L.sd_circle((2.3, 0), c0, 2.0), 0.3))) < 1e-12 and abs(float(L.op_shell(L.sd_circle((1.7, 0), c0, 2.0), 0.3))) < 1e-12))

    # smooth minimum
    aa = np.linspace(-3, 3, 601)
    k = 1.0
    far = np.abs(aa - 0.0) >= k
    s = L.smin(aa, 0.0, k)
    checks.append(("smooth min equals min when |a - b| >= k", bool(np.allclose(s[far], np.minimum(aa, 0.0)[far], atol=1e-12))))
    checks.append(("smooth min is never above min", bool(np.all(s <= np.minimum(aa, 0.0) + 1e-12))))
    checks.append(("smooth min is symmetric in a and b", bool(np.allclose(L.smin(aa, 0.3, k), L.smin(0.3, aa, k)))))
    checks.append(("at a = b the surface is pulled out by exactly k/4", abs(float(L.smin(2.0, 2.0, 0.8)) - (2.0 - 0.2)) < 1e-12))
    checks.append(("edge case: k = 0 gives the plain min", bool(np.allclose(L.smin(aa, 0.5, 0.0), np.minimum(aa, 0.5)))))

    # areas
    R, H, t = 3.852, 9.0, 0.5
    an = L.body_area(R, H, t)
    got = L.area_by_cells(lambda q: L.body2d(q, R, H, t), (-R - 0.5, -0.5), (R + 0.5, H + 0.5), 400)
    checks.append((f"mug body area by counting {got:.3f} is within 0.5% of the analytic {an:.3f} cm^2", abs(got - an) / an < 0.005))
    ring = L.area_by_cells(lambda q: np.abs(L.sd_circle(q, c0, 2.0)) - 0.45, (-3, -3), (3, 3), 600)
    checks.append((f"handle ring area {ring:.3f} is within 0.5% of 4 pi Rh r = {4 * np.pi * 2.0 * 0.45:.3f}", abs(ring - 4 * np.pi * 2.0 * 0.45) / (4 * np.pi * 2.0 * 0.45) < 0.005))

    # the mug cross-section
    checks.append(("mug: the cavity is empty, the wall and the handle tube are solid",
                   float(L.mug2d(np.array([0.0, 5.0]), R, H, t)) > 0
                   and float(L.mug2d(np.array([R - t / 2, 4.0]), R, H, t)) < 0
                   and float(L.mug2d(np.array([R + 3.0, 0.55 * H]), R, H, t)) < 0))
    checks.append(("mug: the hole of the handle is empty", float(L.mug2d(np.array([R + 1.0, 0.55 * H]), R, H, t)) > 0))
    a0 = L.area_by_cells(lambda q: L.mug2d(q, R, H, t, 0.0), (-R - 1, -1), (R + 5, H + 1), 500)
    a1 = L.area_by_cells(lambda q: L.mug2d(q, R, H, t, 1.0), (-R - 1, -1), (R + 5, H + 1), 500)
    checks.append((f"a blend radius of 1 cm adds material at the joints ({a0:.2f} to {a1:.2f} cm^2)", a1 > a0))

    # worked example: 300 mL, H = 9, t = 0.5
    root = np.sqrt(300.0 / (np.pi * 8.5))
    Rd = L.radius_for_capacity(300.0, 9.0, 0.5)
    checks.append((f"worked example: R = 0.5 + sqrt(300 / (pi * 8.5)) = {Rd:.3f} cm and the capacity is 300 mL",
                   abs(Rd - (0.5 + root)) < 1e-12 and abs(Rd - 3.852) < 1e-3 and abs(L.capacity(Rd, 9.0, 0.5) - 300.0) < 1e-9))
    mat = L.material_volume(Rd, 9.0, 0.5)
    rev = L.revolve_volume(lambda q: L.body2d(q, Rd, 9.0, 0.5), Rd, 9.0, 400)
    checks.append((f"volume of revolution: counted slice {rev:.1f} matches the formula {mat:.1f} cm^3 within 1%", abs(rev - mat) / mat < 0.01))
    checks.append(("design numbers: H = 10, t = 0.4 needs R = 3.554; H = 9 needs R = 3.732 (t = 0.4) and 4.213 (t = 0.8)",
                   abs(L.radius_for_capacity(300.0, 10.0, 0.4) - 3.554) < 1e-3 and abs(L.radius_for_capacity(300.0, 9.0, 0.4) - 3.732) < 1e-3
                   and abs(L.radius_for_capacity(300.0, 9.0, 0.8) - 4.213) < 1e-3))
    m4 = L.material_volume(L.radius_for_capacity(300.0, 9.0, 0.4), 9.0, 0.4)
    m8 = L.material_volume(L.radius_for_capacity(300.0, 9.0, 0.8), 9.0, 0.8)
    checks.append((f"design: material is {m4:.1f} cm^3 for t = 0.4 and {m8:.1f} cm^3 for t = 0.8 (and {mat:.1f} for 0.5)",
                   abs(m4 - 93.9) < 0.05 and abs(m8 - 201.7) < 0.05 and abs(mat - 119.5) < 0.05))
    checks.append(("app numbers: R = 4.5, t = 0.5, H = 9 holds 427.3 mL; R = 3.85 holds 299.7 mL; smin(0, 0, 2) = -0.5",
                   abs(L.capacity(4.5, 9.0, 0.5) - 427.3) < 0.05 and abs(L.capacity(3.85, 9.0, 0.5) - 299.7) < 0.05
                   and abs(float(L.smin(0.0, 0.0, 2.0)) + 0.5) < 1e-12))
    checks.append(("a thicker wall needs a larger radius for the same capacity",
                   L.radius_for_capacity(300.0, 9.0, 0.8) > L.radius_for_capacity(300.0, 9.0, 0.4)))
    try:
        L.radius_for_capacity(300.0, 9.0, 9.0)
        ok = False
    except ValueError:
        ok = True
    checks.append(("edge case: a wall as thick as the mug is height is rejected", ok))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
